# Assignment 1: Centroid based clustering

### Make sure you read through this entire notebook before getting started with implementing the algorithm.


This notebook has the following structure:

- We first shortly explain the idea of the assignment.
- We follow thus up with a short walkthrough of the assignment, after which you can start implementing the necessary functions.


# Introduction to this template notebook

* This is a **personal** notebook.
* Make sure you work in a **copy** of `...-template.ipynb`,
**renamed** to `...-yourIDnr.ipynb`,
where `yourIDnr` is your TU/e identification number.

<div class="alert alert-danger" role="danger">
<h3>Integrity</h3>
<ul>
    <li>In this course you must act according to the rules of the TU/e code of scientific conduct.</li>
    <li>All the exercises and the graded assignments are to be executed individually and independently.</li>
    <li>You must not copy from the Internet, your friends, books... If you represent other people's work as your own, then that constitutes fraud and will be reported to the Examination Committee.</li>
    <li>Making your work available to others (complicity) also constitutes fraud.</li>
</ul>
</div>

You are expected to work with Python code in this notebook.

The locations where you should write your solutions can be recognized by
**marker lines**,
which look like this:

>`#//`
>    `BEGIN_TODO [Label]` `Description` `(n points)`
>
>`#//`
>    `END_TODO [Label]`

<div class="alert alert-warning" role="alert">Do NOT modify or delete these marker lines.  Keep them as they are.<br/>
<br/>
NEVER write code <i>outside</i> the marked blocks.
Such code cannot be evaluated.
</div>

Proceed in this notebook as follows:
* **Read** the text.
* **Fill in** your solutions between `BEGIN_TODO` and `END_TODO` marker lines.
* **Run** _all_ code cells (also the ones _without_ your code),
    _in linear order_ from the first code cell.

**Personalize your notebook**:
1. Copy the following three lines of code:

  ```python
  AUTHOR_NAME = 'Your Full Name'
  AUTHOR_ID_NR = '1234567'
  AUTHOR_DATE = 'YYYY-MM-DD'  # when notebook was first modified, e.g. '2020-02-26'
  ```

1. Paste them between the marker lines in the next code cell.
1. Fill in your _full name_, _identification number_, and the current _date_ as strings between quotes.
1. Run the code cell by putting the cursor there and typing **Control-Enter**.


In [1]:
#// BEGIN_TODO [Author] Name, Id.nr., Date, as strings (1 point)

AUTHOR_NAME = 'Trinity Jan'
AUTHOR_ID_NR = '2041839'
AUTHOR_DATE = '2026-05-11'  # when notebook was first modified, e.g. '2020-02-26'

#// END_TODO [Author]

AUTHOR_NAME, AUTHOR_ID_NR, AUTHOR_DATE

('Trinity Jan', '2041839', '2026-05-11')

# Centroid based clustering

For this assignment, you are expected to implement the k_means clustering algorithm, as discussed in class. The functions you are expected to implement are
* ```initialize_centroids```. This function chooses the initial cluster points. Currently, this function already contains an implementation, however, you are strongly encouraged to experiment with creating different initialisation functions once you have implemented the ```cluster``` function.
* ```cluster```. This function should, once implemented by you, perform the actual clustering and find the proper placement of the centroids.

In addition to this file, you are provided two more python files:
* ```point.py```.  This file contains the datastructures ```Point```, ```ClusterPoint``` and ```CentroidPoint``` that you can use in your implementation of the two functions.
* ```dataset.py```. This file contains the tools that read the input from and write the output to file.

<b>Testing.</b> After (partially) implementing the two functions, you can run the <b>run</b> function at the bottom of this document. Just above this function, you are provided with the ```test_case_nr``` field. Changing this field to an integer value $x \in [0, 6]$ allows you to choose which testcase you would like to run. This then reads the file 0$x$.in from the <b>input</b> folder and provides the resulting clustering in 0$x$.out in the <b>output</b> folder.

<b>Visulisation.</b> To view the clustering you created, you can use the ```Visualizer.ipynb``` notebook that you are provided alongside this assignment. After executing the first cell, you can simply execute the cell that corresponds to the testcase you want to view and a visualisation is provided.

<b>Handing in.</b> To verify your implementation, you are expected to hand in this file on Canvas. Here, we use the automated checking tool Momotor to check your implementation of the algorithm. After it has run through all the testcases (should take at most a couple minutes), you can see in the Momotor tab of the course how you scored. 
Note: you should only hand in _this_ file. The visualizer and other python files should not be handed in.

We move on to the actual coding part of this assignment. At the start we import some useful tools.

In [7]:
import sys
import os
import time

from point import *
from dataset import *

""" Runtime parameters """
assignment_nr = 1       # The assignment number. Used by the visualizer to determine what has to be visualized

**Extra functions.** When implementing the ```cluster()``` and ```initialize_centroids()``` functions, you'll likely want to create a couple functions yourself. To make sure the automated grader picks up on these, make sure you place these functions in the below cell.

In [8]:
#// BEGIN_TODO [YOUR-OWN-FUNCTIONS] 

def compute_centroids(cluster_points, n, d, k): 
    """
    Computes the optimal set of centroids for a fixed assignment of cluster labels to the input points
    
    :param cluster_points: cluster points of ClusterPoint class 
    :param n: number of points
    :param d: dimension
    :param k: number of clusters
    """
    
    cluster_sizes = [0] * k # initialize cluster sizes
    point_sum = [ClusterPoint(dimension=d) for _ in range(k)] # initalize point sums 
    centroids = [CentroidPoint(dimension=d) for _ in range(k)] # initialize centroids

    for i in range(n): 
        cluster_sizes[cluster_points[i].cluster_label] += 1 
        point_sum[cluster_points[i].cluster_label].add(other=cluster_points[i])
    print('computed cluster sizes', cluster_sizes)
    print('computed point sum', [i.coords for i in point_sum])
    
    for i in range(k): 
        point_sum[i].div(cluster_sizes[i])
        centroids[i] = point_sum[i]
    print('computed centroids', [i.coords for i in centroids], '\n')
        
    return centroids


def compute_labels(cluster_points, centroids, n, d, k): 
    """
    Assigns cluster points to centroids. 
    
    :param cluster_points: cluster pointsn of ClusterPoint class
    :param centroids: previously defined centroids
    :param n: number of points
    :param d: dimension
    :param k: number of clusters
    """
    
    for i in range(n): 
        min_distance = 999999 # initialize high min distance
        for j in range(k):
            current_distance = cluster_points[i].sq_distance_to(other=centroids[j])
            if current_distance < min_distance: 
                min_distance = current_distance
                cluster_label = j 
        cluster_points[i].cluster_label = cluster_label
    
    return cluster_points

def read(path_in):
    """
    Reads the input set, clusters the points and writes to output

    :param path_in:     location of the input set
    :param path_out:    location to print the output
    """

    # read input from file
    try:
        with open(path_in, "r") as f:
            input_obj = Dataset.read_input(f)
            return input_obj
    except IOError:
        print("Could not read input file: " + path_in, file=sys.stderr)
        return None    

#// END_TODO [YOUR-OWN-FUNCTIONS]

In [10]:
input_obj = read('./input/00.in')

In [5]:
input_obj.n # number of points

500

In [6]:
input_obj.d # dimensions

2

In [7]:
input_obj.k # number of clusters

4

In [8]:
cluster_points = input_obj.cluster_points # cluster points 
cluster_point = cluster_points[0] # cluster point class

In [9]:
cluster_point.dimension # dimensions

2

In [10]:
cluster_point.coords # coordinates

[34.72993170400471, -4.8425522047457985]

In [11]:
cluster_point.cluster_label # label initialized to -1

-1

**Initialize centroids.** We continue with the ```intialize_centroids()``` function. This function currently already has a basic implementation: it takes the first $k$ cluster points from the input and sets these as initial centroids. When checking your algorithm implementations in Momotor, only this basic implementation will be used. However, for the report you are expected to write about this assignment, you are very much encouraged to overwrite this basic implementation with something different and report on your findings.

In [80]:
def initialize_centroids(input_obj):
    """
    Calculates the initial centroid placement

    :param input_obj:   the input object
    :return:            a list of k centroids in the plane
    """
    ### IMPLEMENTATION: FIRST K CLUSTER POINTS AS INITIAL CENTROIDS
    centroids = [CentroidPoint(p.dimension, list(p.coords)) for p in input_obj.cluster_points[:input_obj.k]]

#// BEGIN_TODO [IMPLEMENT-INITIALIZE-CENTROIDS]
    
    ### IMPLEMENTATION: GONZALES
    cluster_points = input_obj.cluster_points
    centroids = [CentroidPoint(dimension=input_obj.d) for _ in range(input_obj.k)] # initialize centroids # TODO not needed
    min_distances = [0] * input_obj.n
    
    # initialize first centroids with first input point
    centroids[0] = cluster_points[0]
    
    for j in range(input_obj.n): 
        min_distances[j] = 999999 # initialize high min distances
    
    for i in range(1, input_obj.k): 
        # update distance to closest centroid for each input point
        for j in range(input_obj.n): 
            x = cluster_points[j].sq_distance_to(other=centroids[i-1])
            if min_distances[j] > x: 
                min_distances[j] = x 
        
        # compute input point which is farthest from current set of centroids
        max_distance = 0 # initialize lowest max distance
        
        for j in range(input_obj.n): 
            if max_distance < min_distances[j]: 
                max_distance = min_distances[j]
                farthest_point = cluster_points[j]
                    
        # add farthest point to set of centroids
        centroids[i] = farthest_point
    
    
    ### IMPLEMENTATION KMEANS++

#// END_TODO [IMPLEMENT-INITIALIZE-CENTROIDS]
    return centroids

In [57]:
initialized_centroids = initialize_centroids(input_obj)

In [58]:
for ic in initialized_centroids: 
    print(ic.coords)

[34.72993170400471, -4.8425522047457985]
[-100.51130517668257, 17.758233484720094]
[-35.563227199157055, -30.91974940200617]
[31.238946081426253, -77.19505801610241]


**Cluster.** Below we find the ```cluster()``` function. Currently, no implementations has been provided. It is your task to implement the k-means clustering algorithm here.

In [81]:
def cluster(input_obj):
    """
    Perform k-means clustering on the input set

    :param input_obj:   the input object
    :return:            a list of k centroids in the plane
    """    
#// BEGIN_TODO [IMPLEMENT-CLUSTER]    
    
    # initalize centroids
    centroids_old = initialize_centroids(input_obj)
    
    # TODO: can remove the n, d, k because can do in each iteration again or NAHH
    # obtain n, d, k once
    # initialize cluster points
    n = input_obj.n
    d = input_obj.d
    k = input_obj.k
    cluster_points = input_obj.cluster_points
    
    # repeat until centroids did not change in the last iteration
    i = 0 
    while True: 
        print('iteration', i)
        
        cluster_points = compute_labels(cluster_points, centroids_old, n, d, k)
        centroids_new = compute_centroids(cluster_points, n, d, k)
        
        # stop if unchanged
        if centroids_new == centroids_old:
            break
        
        centroids_old = centroids_new
        
        i+=1
        
    centroids = centroids_new

#// END_TODO [IMPLEMENT-CLUSTER] 
    return centroids

In [82]:
cluster(input_obj)

iteration 0
computed cluster sizes [128, 125, 125, 122]
computed point sum [[5101.898850147319, 669.2200041249636], [-9874.638912272108, 2633.15185549266], [-3238.9416075856207, -1970.046153586201], [3690.7646193185383, -6875.447427773945]]
computed centroids [[39.85858476677593, 5.228281282226278], [-78.99711129817686, 21.06521484394128], [-25.911532860684964, -15.760369228689608], [30.25216901080769, -56.356126457163484]] 

iteration 1
computed cluster sizes [125, 125, 125, 125]
computed point sum [[5003.25037041182, 783.0317646367322], [-9874.638912272108, 2633.15185549266], [-3238.9416075856207, -1970.046153586201], [3789.413099054038, -6989.259188285713]]
computed centroids [[40.02600296329456, 6.2642541170938575], [-78.99711129817686, 21.06521484394128], [-25.911532860684964, -15.760369228689608], [30.315304792432304, -55.9140735062857]] 

iteration 2
computed cluster sizes [125, 125, 125, 125]
computed point sum [[5003.25037041182, 783.0317646367322], [-9874.638912272108, 2633.1

Next, we define the function that will take the input from file, use your clustering algorithm on get the clustering and write the result to file again.

In [72]:
def run(path_in, path_out):
    """
    Reads the input set, clusters the points and writes to output

    :param path_in:     location of the input set
    :param path_out:    location to print the output
    """

    # read input from file
    try:
        with open(path_in, "r") as f:
            input_obj = Dataset.read_input(f)
    except IOError:
        print("Could not read input file: " + path_in, file=sys.stderr)
        return        

    # find the best centroids using k_means
    centroids = cluster(input_obj)

    # simple check if the correct number of centroids has been given
    assert len(centroids) == input_obj.k

    # print result to file
    try:
        input_obj.write_output(centroids, path_out, assignment_nr)
    except IOError:
        print("Could not write output to file: " + path_out, file=sys.stderr)

    print("Cluster counter:      ", input_obj.k, file=sys.stderr)
    print("Mean squared distance: {:.3f}".format(input_obj.avg_score(centroids)), file=sys.stderr)                

**Running testcases.** Lastly, you can check your implementation by running some tests on it. Below, you can choose which testcase ```test_case_nr``` $\in[0,6]$ you would like to run.

In [73]:
test_case_nr =   0     # which input file to read

if __name__ == "__main__":
    start = time.time()
    run("input/{:02d}.in".format(test_case_nr), "output/{:02d}.out".format(test_case_nr))
    end = time.time()
    print("Time taken:            {:.3f}s".format(end - start), file=sys.stderr)   

iteration 0
computed cluster sizes [175, 195, 59, 71]
computed point sum [[4058.409345626902, -87.63597997125798], [-12059.86883222167, 1695.701086357107], [1241.4868493105748, -2892.7950812167605], [2439.0555868923166, -4258.391746911611]]
computed centroids [[23.19091054643944, -0.5007770284071885], [-61.845481190880356, 8.695903006959522], [21.04214998831483, -49.03042510536882], [34.35289559003263, -59.97734854805086]] 

iteration 1
computed cluster sizes [154, 207, 62, 77]
computed point sum [[4519.140109682907, 359.6361260345882], [-12355.983813853854, 1470.793150148549], [915.5155925106849, -2777.1078207962005], [2600.41106126839, -4596.4431771294585]]
computed centroids [[29.3450656472916, 2.335299519705118], [-59.69074306209591, 7.105280918592024], [14.766380524365886, -44.79206162574517], [33.7715722242648, -59.69406723544751]] 

iteration 2
computed cluster sizes [129, 209, 60, 102]
computed point sum [[4953.662708774878, 768.3354333586345], [-12381.345132337407, 1483.630747

Cluster counter:       4
Mean squared distance: 137.727
Time taken:            0.055s


That is all. At this point you should have all the information you should need to get started. If you have any questions you are free to ask the tutor overseeing the class. If you are experiencing issues with handing in your submission to Momotor, you can contact the responsible teaching assistant via email. Their email address can be found on Canvas.

Best of luck and happy clustering!

&copy; 2019-2020 - **TU/e** - Eindhoven University of Technology